# Merge gemma_ksl_lora into Gemma 4 E4B + export ksl_model.litertlm

## Setup (do this ONCE before running any cell)

1. **Add the LoRA as a dataset.** Sidebar -> `+ Add Input` -> `Upload Dataset`. Upload all 7 files from `mobile-app/sema/sema/Models/gemma_ksl_lora/`:
   `adapter_config.json`, `adapter_model.safetensors`, `chat_template.jinja`, `processor_config.json`, `README.md`, `tokenizer_config.json`, `tokenizer.json`.
   The dataset slug doesn't matter — Cell 4 scans `/kaggle/input/` to find it.
2. **Accelerator + Internet.** Sidebar -> `Settings`:
   - Accelerator: `GPU T4 x2` or `GPU P100`
   - Internet: `ON`
3. **Persistence.** Sidebar -> `Settings` -> `Persistence` -> `Files only`. So the merged checkpoint and the .litertlm survive a Restart Session.

## How to run

- Run **Cell 1** (install). Wait ~3 min.
- When prompted, **Run -> Restart Session**.
- Run **Cells 2 through 5** in order.
- Download `/kaggle/working/ksl_model.litertlm` from the Output panel.
- Drop it into `mobile-app/sema/sema/Resources/`.

Total runtime ~15 min end-to-end.

## Cell 1 — Install pinned dependencies

Kaggle's preinstalled stack (cudf, cuml, torchvision, torchaudio, etc.) pins old/incompatible torch versions and breaks `litert-torch`. We uninstall those first, then force-install the exact pin set known to work locally. The `--no-deps` flag on torch/torchao stops pip from "helpfully" trying to satisfy the CUDA-stack constraints we don't care about.

In [ ]:
import subprocess, sys

# Wipe the Kaggle CUDA stack that fights every pin.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q',
    'torchaudio', 'torchvision', 'timm',
    'cudf-cu12', 'cuml-cu12', 'libcuml-cu12', 'libcuvs-cu12', 'libraft-cu12',
    'cuda-python', 'cuda-bindings', 'bigframes', 'google-adk',
], check=False)
# timm is removed because litert-torch's gemma3n patch imports
# transformers.models.gemma3n, which optionally imports timm, which
# requires torchvision (which we also remove). With timm gone,
# transformers' is_timm_available() returns False and that import path
# is skipped entirely.

# Pin torch and torchao with --no-deps so nothing pulls them sideways.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
    '--upgrade', '--force-reinstall', '--no-deps',
    'torch==2.11.0',
    'torchao==0.17.0',
    'transformers==5.8.1',
    'tokenizers==0.22.2',
    'accelerate==1.13.0',
    'huggingface_hub==1.15.0',
    'peft==0.19.1',
])

# litert-torch and friends bring their own (correct) deps.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
    'litert-torch==0.9.0',
    'ai-edge-litert==2.1.4',
    'ai-edge-quantizer==0.6.0',
    'safetensors',
    'protobuf>=6.31.1',  # litert-torch's gencode targets protobuf 6.x;
                         # Kaggle preinstalls 5.29.5 which crashes at import.
    'filelock', 'requests', 'pyyaml', 'regex', 'tqdm', 'packaging',
])

print('\n' + '='*64)
print('INSTALL DONE. Now do:  Kaggle menu -> Run -> Restart Session')
print('Then run Cells 2 through 5 in order.')
print('='*64)

## Cell 2 — Verify imports (after restart)

Surfaces the real error if anything in the pin set didn't take. Should print:
```
torch=2.11.0+cu130  ...
transformers=5.8.1
litert-torch import OK
All imports OK. Proceed to Cell 3.
```
If you see anything else, paste the full traceback back.

In [ ]:
import traceback

# Force-disable the timm code path in transformers BEFORE we trigger
# litert-torch's import (which loads transformers.models.gemma3n, which
# optionally pulls in timm -> torchvision and breaks on Kaggle). Stale
# dist-info for an uninstalled timm keeps is_timm_available() returning
# True, so we override the bool and the function directly.
import transformers
import transformers.utils
import transformers.utils.import_utils as _iu
_iu._timm_available = False
_iu.is_timm_available = lambda: False
transformers.utils.is_timm_available = lambda: False

import torch
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}')
assert torch.__version__.split('+', 1)[0].startswith('2.11.'), (
    f'torch is {torch.__version__}, expected 2.11.x. Did you restart the session?')
print(f'transformers={transformers.__version__}')

try:
    from transformers.modeling_utils import AttentionInterface
    print('AttentionInterface direct import OK')
except Exception:
    print('AttentionInterface FAILED -- full traceback below:')
    traceback.print_exc()
    raise

from torchao.quantization import Granularity
print('torchao.quantization.Granularity OK')

from litert_torch.generative.export_hf.export import export as run_export
print('litert-torch import OK')

print('\nAll imports OK. Proceed to Cell 3.')


## Cell 3 — Find the LoRA dataset

In [ ]:
from pathlib import Path

lora_dir = None
for found in Path('/kaggle/input').rglob('adapter_model.safetensors'):
    if (found.parent / 'adapter_config.json').exists():
        lora_dir = found.parent
        break
assert lora_dir is not None, (
    'No adapter_model.safetensors found under /kaggle/input. '
    'Did you upload the gemma_ksl_lora folder as a Kaggle dataset?')
print(f'LoRA dir: {lora_dir}')
for f in sorted(lora_dir.iterdir()):
    sz = f.stat().st_size
    print(f'  {f.name:40s}  {sz/1024/1024:.1f} MB' if sz > 1e6 else f'  {f.name}')

## Cell 4 — Merge LoRA into base Gemma 4 E4B

Skips automatically if a complete merged checkpoint already exists at `/kaggle/working/gemma_ksl_merged/` (so re-running this cell after an interrupt is safe). ~3 min on T4 GPU.

In [ ]:
import gc, json, shutil, time
from pathlib import Path
import torch
from transformers import Gemma4ForConditionalGeneration
from peft import PeftModel, LoraConfig

BASE = 'unsloth/gemma-4-E4B-it'
MERGED_DIR = Path('/kaggle/working/gemma_ksl_merged')

merged_ok = (MERGED_DIR / 'model.safetensors.index.json').exists() and any(
    MERGED_DIR.glob('model-*.safetensors'))
if merged_ok:
    print(f'Merge already done at {MERGED_DIR}, skipping.')
else:
    print(f'[1] Loading base {BASE} ...')
    t0 = time.time()
    model = Gemma4ForConditionalGeneration.from_pretrained(
        BASE, torch_dtype=torch.bfloat16, device_map='auto', low_cpu_mem_usage=True)
    print(f'    base loaded in {time.time()-t0:.0f}s')

    print(f'[2] Attaching LoRA (text decoder only) ...')
    cfg = json.load(open(lora_dir / 'adapter_config.json'))
    # Stock PEFT can't graft onto Gemma4ClippableLinear in the vision/audio
    # towers (Unsloth patches PEFT for that). Skip those branches — text-only
    # inference doesn't need them.
    cfg['exclude_modules'] = '(.*vision_tower.*)|(.*audio_tower.*)'
    for k in ('alora_invocation_tokens', 'auto_mapping', 'arrow_config',
              'corda_config', 'ensure_weight_tying', 'eva_config',
              'qalora_group_size', 'trainable_token_indices', 'use_qalora'):
        cfg.pop(k, None)
    peft_config = LoraConfig(**cfg)
    model = PeftModel.from_pretrained(model, str(lora_dir), config=peft_config)

    print('[3] Merging (merge_and_unload) ...')
    model = model.merge_and_unload()

    print(f'[4] Saving merged model to {MERGED_DIR} ...')
    MERGED_DIR.mkdir(parents=True, exist_ok=True)
    t3 = time.time()
    model.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size='4GB')
    for name in ('tokenizer.json', 'tokenizer_config.json',
                 'chat_template.jinja', 'processor_config.json'):
        src = lora_dir / name
        if src.exists():
            shutil.copy2(src, MERGED_DIR / name)
    print(f'    saved in {time.time()-t3:.0f}s')

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print('\nMerge step done.')

## Cell 5 — Export merged model to .litertlm (INT4)

This is the heavy step (~5-10 min). Skips automatically if `ksl_model.litertlm` already exists in `/kaggle/working/`.

In [ ]:
import shutil, time
from pathlib import Path
from litert_torch.generative.export_hf.export import export as run_export

MERGED_DIR = Path('/kaggle/working/gemma_ksl_merged')
EXPORT_DIR = Path('/kaggle/working')
OUTPUT_NAME = 'ksl_model'
final = EXPORT_DIR / f'{OUTPUT_NAME}.litertlm'

if final.exists():
    print(f'{final} already exists ({final.stat().st_size/1024/1024:.1f} MB). Done.')
else:
    print('Exporting to .litertlm (INT4 dynamic, text-only) ...')
    t = time.time()
    run_export(
        model=str(MERGED_DIR),
        output_dir=str(EXPORT_DIR),
        quantization_recipe='dynamic_wi4_afp32',
        export_vision_encoder=False,    # text-only
        externalize_embedder=True,      # required for Gemma 4 PLE arch
        bundle_litert_lm=True,          # single-file .litertlm
    )
    print(f'export finished in {time.time()-t:.0f}s')

    cands = sorted(EXPORT_DIR.glob('*.litertlm'),
                   key=lambda p: p.stat().st_mtime, reverse=True)
    assert cands, 'Export claimed success but no .litertlm landed.'
    src = cands[0]
    if src != final:
        src.rename(final)
    print(f'\nartefact: {final}  ({final.stat().st_size/1024/1024:.1f} MB)')

    # Free the 15 GB merged checkpoint -- not needed after the export.
    shutil.rmtree(MERGED_DIR, ignore_errors=True)

print('\nDone. Download ksl_model.litertlm from the Kaggle Output panel and')
print('drop it into mobile-app/sema/sema/Resources/.')